# Huấn luyện bộ phân loại cử chỉ tay

**Đề tài:** AI Hand-Gesture Authenticated Smart Door Lock — hệ thống IoT 4 tầng

Notebook này huấn luyện một mạng nơ-ron nhỏ (MLP) để phân loại **7 lớp cử chỉ**
từ vector đặc trưng **42 chiều** do `common.preprocess_landmarks()` sinh ra.

| Bước | Nội dung |
|---|---|
| 1 | Nạp thư viện và cấu hình |
| 2 | Đọc `model/keypoint.csv` + `model/keypoint_meta.csv` và kiểm tra dữ liệu |
| 3 | Chia train/validation **theo burst** — không burst nào nằm ở cả hai phía |
| 4 | Dựng mạng MLP |
| 5 | Huấn luyện với EarlyStopping |
| 6 | Đánh giá: classification report + ma trận nhầm lẫn |
| 7 | Độ chính xác từng lớp và cảnh báo cặp lớp dễ nhầm — **con số báo cáo** |
| 8 | Train lại trên **toàn bộ** dữ liệu — **model triển khai** |
| 9 | Lưu model `.keras` và `.tflite`, đối chiếu hai bản |

> **Con số báo cáo và model triển khai là hai thứ khác nhau.** Mọi con số đánh giá
> (Bước 6–7) đến từ lần chia theo burst. Model được lưu ra `.tflite` (Bước 9) là bản
> train lại trên toàn bộ dữ liệu ở Bước 8 — bản đó không có con số đánh giá riêng.

> **Chạy notebook này TRƯỚC KHI** chạy `gesture_lock.py`, vì `gesture_lock.py`
> nạp file `.tflite` do bước 7 sinh ra.
>
> Yêu cầu: đã chạy `collect_data.py` và thu **250–300 mẫu mỗi lớp**, kể cả lớp `none`.

## Bước 1 — Nạp thư viện và cấu hình

Mọi tham số (đường dẫn, danh sách lớp) đều đọc từ `config.json`, không viết cứng
trong notebook. Nhờ vậy sửa một chỗ là cả hệ thống đổi theo.

In [ ]:
import csv
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

# Cho phép import common.py nằm cùng thư mục vision/
VISION_DIR = Path.cwd()
if not (VISION_DIR / "common.py").exists():          # phòng khi mở notebook từ gốc dự án
    VISION_DIR = VISION_DIR / "vision"
sys.path.insert(0, str(VISION_DIR))

from common import load_config, gesture_tables, resolve_path, FEATURE_DIM

cfg = load_config(VISION_DIR / "config.json")
ID_TO_LABEL, _ = gesture_tables(cfg)
CLASS_NAMES = [ID_TO_LABEL[i] for i in sorted(ID_TO_LABEL)]
NUM_CLASSES = len(CLASS_NAMES)

CSV_PATH     = resolve_path(cfg["model"]["keypoint_csv"])
META_PATH    = resolve_path(cfg["model"]["keypoint_meta_csv"])
KERAS_PATH   = resolve_path(cfg["model"]["classifier_keras"])
TFLITE_PATH  = resolve_path(cfg["model"]["classifier_tflite"])
CM_PNG_PATH  = (VISION_DIR.parent / "docs" / "confusion_matrix.png").resolve()

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow :", tf.__version__)
print("So chieu dac trung :", FEATURE_DIM)
print("So lop             :", NUM_CLASSES, "->", CLASS_NAMES)
print("File du lieu       :", CSV_PATH)

# ===================================================================
#  CO TRON HaGRID VAO TAP TRAIN?  —  MAC DINH PHAI LA False
# ===================================================================
#  False (mac dinh):
#     HaGRID la TAP KIEM THU DOC LAP. Model chi hoc tu du lieu tu thu bang
#     webcam, va vision/eval_external.py do duoc dung cau hoi:
#     "model co nhan duoc ban tay cua NGUOI KHAC khong".
#
#  True:
#     HaGRID duoc noi vao tap train. KHI DO HaGRID KHONG CON LA TAP KIEM THU
#     DOC LAP NUA — con so cua eval_external.py mat y nghia, vi model da nhin
#     thay chinh du lieu do trong luc hoc.
#     Bat co nay thi PHAI ghi ro trong bao cao, va TUYET DOI khong duoc trinh
#     bay con so HaGRID nhu ket qua kiem thu doc lap.
USE_HAGRID_FOR_TRAINING = False

HAGRID_CSV = resolve_path("model/keypoint_hagrid.csv")
print("USE_HAGRID_FOR_TRAINING :", USE_HAGRID_FOR_TRAINING)


### Về cờ `USE_HAGRID_FOR_TRAINING`

Cờ này nằm ở cell đầu và **mặc định là `False`**.

HaGRID được dùng làm **tập kiểm thử độc lập**: model học từ dữ liệu tự thu bằng webcam,
rồi `vision/eval_external.py` đo xem nó có nhận được bàn tay của *người khác* hay không.
Con số trên HaGRID sẽ thấp hơn tập validation tự thu — đó là kết quả đúng và trung thực,
không phải lỗi. Khoảng cách giữa hai con số chính là thứ đáng đưa vào báo cáo.

Nếu bật cờ lên `True`, HaGRID được nối vào tập train. **Khi đó HaGRID không còn là tập
kiểm thử độc lập nữa** — model đã nhìn thấy chính dữ liệu đó lúc học, nên con số của
`eval_external.py` mất ý nghĩa kiểm thử. Bật cờ thì phải ghi rõ trong báo cáo.

## Bước 2 — Đọc dữ liệu và kiểm tra

Định dạng `keypoint.csv`: mỗi dòng là 1 mẫu, **cột 1** là `class_id`, **42 cột sau**
là vector đặc trưng.

Bốn thứ cần kiểm tra trước khi train, vì nếu sai thì train xong mới phát hiện sẽ rất mất công:

1. **Số cột đúng 43** — sai nghĩa là khâu tiền xử lý đã đổi mà dữ liệu cũ còn lẫn vào.
2. **Không có NaN** — một dòng NaN đủ làm toàn bộ loss thành NaN.
3. **Các lớp cân bằng** — lớp nào quá ít mẫu thì model gần như không học được lớp đó.
4. **`keypoint_meta.csv` khớp từng dòng** — mỗi mẫu phải biết mình thuộc burst nào
   (`burst_id`) thì mới chia train/validation theo burst được. File này do
   `collect_data.py` ghi song song với `keypoint.csv`.

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Chua co {CSV_PATH}.\n"
        f"Hay chay thu du lieu truoc:\n"
        f'  D:\\IoT_Gesture_sequence_door_lock\\.conda\\python.exe vision\\collect_data.py'
    )

data = np.loadtxt(CSV_PATH, delimiter=",", dtype=np.float32)
if data.ndim == 1:                      # phong truong hop file chi co 1 dong
    data = data.reshape(1, -1)

X = data[:, 1:]
y = data[:, 0].astype(np.int32)

print("Kich thuoc file :", data.shape)

# --- Kiem tra 1: so cot ---
assert data.shape[1] == FEATURE_DIM + 1, (
    f"Moi dong phai co {FEATURE_DIM + 1} cot (1 nhan + {FEATURE_DIM} dac trung), "
    f"dang co {data.shape[1]}. Co the file CSV lan du lieu cu."
)

# --- Kiem tra 2: NaN ---
n_nan = int(np.isnan(X).sum())
assert n_nan == 0, f"Co {n_nan} gia tri NaN trong du lieu. Xoa cac dong hong roi chay lai."

# --- Kiem tra 4: keypoint_meta.csv khop tung dong voi keypoint.csv ---
if not META_PATH.exists():
    raise FileNotFoundError(
        f"Chua co {META_PATH}.\n"
        "File nay ghi session_id, burst_id cho tung mau; thieu no thi khong chia\n"
        "train/val theo burst duoc. collect_data.py tu ghi file nay khi thu du lieu."
    )
with open(META_PATH, newline="", encoding="utf-8") as f:
    meta_rows = list(csv.DictReader(f))
assert len(meta_rows) == len(y), (
    f"keypoint_meta.csv co {len(meta_rows)} dong du lieu, keypoint.csv co {len(y)} dong. "
    "Hai file phai khop tung dong - co the da sua tay mot file ma quen file kia."
)
groups = np.array([r["burst_id"] for r in meta_rows])
sessions = np.array([r["session_id"] for r in meta_rows])

# --- Tuy chon: noi HaGRID vao tap train (xem canh bao o cell dau) ---
if USE_HAGRID_FOR_TRAINING:
    if not HAGRID_CSV.exists():
        raise FileNotFoundError(
            f"USE_HAGRID_FOR_TRAINING=True nhung chua co {HAGRID_CSV}.\n"
            "Chay kaggle/extract_hagrid_landmarks.ipynb tren Kaggle roi tai file ve."
        )
    dh = np.loadtxt(HAGRID_CSV, delimiter=",", dtype=np.float32)
    if dh.ndim == 1:
        dh = dh.reshape(1, -1)
    assert dh.shape[1] == FEATURE_DIM + 1, (
        f"keypoint_hagrid.csv phai co {FEATURE_DIM + 1} cot, dang co {dh.shape[1]}"
    )
    n_truoc = len(y)
    X = np.vstack([X, dh[:, 1:]])
    y = np.concatenate([y, dh[:, 0].astype(np.int32)])
    # Moi dong HaGRID la mot anh doc lap -> moi dong mot nhom rieng.
    groups = np.concatenate([groups, np.array([f"hagrid_{i}" for i in range(len(dh))])])
    sessions = np.concatenate([sessions, np.full(len(dh), "hagrid")])
    print()
    print("*" * 70)
    print("*  DA TRON HaGRID VAO TAP TRAIN                                      *")
    print(f"*  {n_truoc} mau tu thu + {len(dh)} mau HaGRID = {len(y)} mau" .ljust(69) + "*")
    print("*  HaGRID KHONG con la tap kiem thu doc lap. Con so cua              *")
    print("*  eval_external.py khong con y nghia kiem thu. Ghi ro vao bao cao.  *")
    print("*" * 70)
elif HAGRID_CSV.exists():
    print(f"(Co {HAGRID_CSV.name} nhung USE_HAGRID_FOR_TRAINING=False")
    print(" -> giu lam tap kiem thu doc lap, dung cho eval_external.py)")

# --- Kiem tra 3: phan bo lop ---
print()
print(f"{'id':<4}{'lop':<12}{'so mau':>8}{'ty le':>9}{'burst':>7}{'phien':>7}")
print("-" * 48)
for cid in sorted(ID_TO_LABEL):
    m = (y == cid)
    n = int(m.sum())
    canh_bao = "  <-- qua it" if n < 100 else ""
    print(f"{cid:<4}{ID_TO_LABEL[cid]:<12}{n:>8}{n / len(y):>8.1%}"
          f"{len(set(groups[m])):>7}{len(set(sessions[m])):>7}{canh_bao}")
print("-" * 48)
print(f"{'':<16}{len(y):>8}  tong cong, {len(set(groups))} burst, {len(set(sessions))} phien")

thieu = [ID_TO_LABEL[c] for c in sorted(ID_TO_LABEL) if (y == c).sum() == 0]
if thieu:
    raise ValueError(f"Chua co mau nao cho lop: {thieu}. Phai thu du 7 lop moi train duoc.")

## Bước 3 — Chia train / validation theo burst

Mỗi lần bấm **R** trong `collect_data.py` ghi 30 mẫu liên tiếp, cách nhau 2 khung hình —
gần như cùng một tư thế tay. Nếu chia ngẫu nhiên từng mẫu (`train_test_split`), 30 mẫu đó
bị xé ra hai phía: ở validation model gặp lại đúng tư thế nó vừa học ở train, và điểm
validation bị **thổi phồng** mà không có dấu hiệu gì báo ra ngoài.
(Phép đo cụ thể: `scripts/chan_doan_du_lieu.py`, mục 3b — cột "tính cả burst" so với "BỎ burst".)

Vì vậy chia theo **`burst_id`** đọc từ `keypoint_meta.csv`: mọi mẫu của một burst nằm
trọn ở train **hoặc** trọn ở validation.

**Chia riêng trong từng lớp.** Với mỗi lớp, `GroupShuffleSplit(test_size=0.20,
random_state=42)` chọn khoảng 20% số burst của lớp đó làm validation. Không chạy
`GroupShuffleSplit` một lần trên toàn bộ dữ liệu vì nó **không phân tầng theo lớp**: trên
dữ liệu ngày 23/10 (86 burst), chạy một lần với `random_state=42` thì lớp `palm` không có
burst nào trong validation, và thử 1000 seed thì 30% seed để trống ít nhất một lớp. Mỗi
burst chỉ thuộc đúng một lớp, nên chia trong từng lớp vẫn giữ nguyên nguyên tắc nhóm.

`random_state=42` để mỗi lần chạy lại đều ra cùng một cách chia.

**Giới hạn:** chia theo burst đo được "model có nhận ra một *lần làm cử chỉ mới* không",
nhưng các burst cùng một phiên vẫn chung ánh sáng và chỗ ngồi. Khi đã có từ 3 phiên thu
mới trở lên, chia theo `session_id` mới đo được khả năng sang *điều kiện mới*.

In [ ]:
# Chia theo burst, RIENG TUNG LOP (ly do: xem markdown phia tren).
# Moi burst chi thuoc mot lop, nen chia trong tung lop van giu dung nguyen tac:
# khong burst nao co mau nam o ca train lan validation.
idx_train, idx_val = [], []
for cid in sorted(ID_TO_LABEL):
    idx_c = np.where(y == cid)[0]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    tr, va = next(gss.split(X[idx_c], y[idx_c], groups[idx_c]))
    idx_train.append(idx_c[tr])
    idx_val.append(idx_c[va])
idx_train = np.concatenate(idx_train)
idx_val = np.concatenate(idx_val)

X_train, X_val = X[idx_train], X[idx_val]
y_train, y_val = y[idx_train], y[idx_val]
g_train, g_val = groups[idx_train], groups[idx_val]

# --- Kiem tra: khong burst nao nam o ca hai phia ---
chung = set(g_train) & set(g_val)
assert not chung, f"Co {len(chung)} burst nam o ca train va val: {sorted(chung)[:5]}"

print("train:", X_train.shape, " validation:", X_val.shape)
print(f"burst: {len(set(g_train))} train / {len(set(g_val))} val, khong burst nao nam o ca hai phia")
print()
print(f"{'lop':<12}{'train':>8}{'val':>8}{'burst train':>13}{'burst val':>11}")
print("-" * 52)
for cid in sorted(ID_TO_LABEL):
    mt, mv = (y_train == cid), (y_val == cid)
    print(f"{ID_TO_LABEL[cid]:<12}{int(mt.sum()):>8}{int(mv.sum()):>8}"
          f"{len(set(g_train[mt])):>13}{len(set(g_val[mv])):>11}")


## Bước 4 — Dựng mạng MLP

```
Input(42)
  -> Dropout(0.2)
  -> Dense(20, relu)
  -> Dropout(0.4)
  -> Dense(10, relu)
  -> Dense(7, softmax)
```

Mạng **cố tình làm rất nhỏ** (chỉ ~1 nghìn tham số). Lý do:

- Đầu vào đã được chuẩn hoá kỹ ở `preprocess_landmarks()` (bỏ vị trí, bỏ kích thước,
  quy tay trái về tay phải), nên bài toán còn lại khá dễ — không cần mạng lớn.
- Dữ liệu chỉ vài nghìn mẫu. Mạng lớn sẽ **học thuộc** dữ liệu thay vì học quy luật.
- Mạng nhỏ chạy được ở tốc độ camera thời gian thực trên CPU laptop.

Hai lớp `Dropout` chính là để chống học thuộc: mỗi bước huấn luyện ngẫu nhiên tắt bớt
nơ-ron, buộc mạng không được phụ thuộc vào một vài đặc trưng riêng lẻ.

Mạng được dựng trong hàm `dung_model()` để Bước 8 dựng lại **đúng kiến trúc, đúng seed**
khi train bản triển khai.

Dùng `sparse_categorical_crossentropy` vì nhãn đang ở dạng số nguyên `0..6`
(nếu dùng `categorical_crossentropy` thì phải one-hot hoá trước).

In [ ]:
def dung_model():
    """Dung va compile mot MLP moi tu dau. Buoc 8 goi lai ham nay de train ban
    trien khai voi CUNG kien truc va CUNG seed."""
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(FEATURE_DIM,)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(20, activation="relu"),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(10, activation="relu"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ])
    m.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return m


model = dung_model()
model.summary()

## Bước 5 — Huấn luyện

`EarlyStopping(patience=20)` theo dõi `val_loss`: nếu qua **20 epoch liên tiếp** mà
validation không khá lên thì dừng, và `restore_best_weights=True` lấy lại trọng số
của epoch tốt nhất chứ không giữ trọng số của epoch cuối.

Nhờ đó đặt `epochs=1000` cũng không sao — thực tế sẽ tự dừng sớm hơn nhiều.

Số epoch tốt nhất được ghi lại vào `SO_EPOCH_TOT_NHAT`: bản triển khai ở Bước 8 train
trên toàn bộ dữ liệu, không còn validation để dừng sớm, nên chạy đúng chừng đó epoch.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=1000,
    batch_size=128,
    callbacks=[early_stop],
    verbose=2,
)

# restore_best_weights lay trong so cua epoch co val_loss nho nhat -> ghi lai so epoch do.
SO_EPOCH_TOT_NHAT = int(np.argmin(history.history["val_loss"])) + 1

print()
print("So epoch da chay :", len(history.history["loss"]))
print("Epoch tot nhat   :", SO_EPOCH_TOT_NHAT, "(dung lai o Buoc 8)")
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"val_loss = {val_loss:.4f}   val_accuracy = {val_acc:.4f}")

### Đường học

Hai đường `train` và `validation` bám sát nhau là tốt. Nếu đường train tiếp tục giảm
mà đường validation đi ngang rồi ngóc lên, đó là dấu hiệu **học thuộc** — cần thu thêm
dữ liệu đa dạng hơn (đổi góc tay, đổi ánh sáng, đổi khoảng cách).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.show()

## Bước 6 — Đánh giá trên tập validation (chia theo burst)

`classification_report` cho 3 con số mỗi lớp:

- **precision** — trong những lần model kêu là lớp X, bao nhiêu phần trăm đúng.
  Thấp nghĩa là model hay **báo nhầm** thành lớp X.
- **recall** — trong những mẫu thật sự là lớp X, model bắt được bao nhiêu phần trăm.
  Thấp nghĩa là model hay **bỏ sót** lớp X.
- **f1-score** — trung bình điều hoà của hai cái trên.

Với bài khoá cửa này, **precision của các lớp cử chỉ quan trọng hơn recall**: báo nhầm
một cử chỉ sẽ làm sai mã và bị khoá, còn bỏ sót thì người dùng chỉ cần giữ tay lâu hơn.

In [ ]:
y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

print(classification_report(
    y_val, y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    digits=3,
    zero_division=0,
))

### Ma trận nhầm lẫn

Đọc theo **hàng**: hàng `i` là các mẫu thật sự thuộc lớp `i`, cột `j` là số mẫu bị
model đoán thành lớp `j`. Đường chéo là đoán đúng. Ô sáng nằm ngoài đường chéo chính
là **cặp cử chỉ bị nhầm** — đó là chỗ cần thu thêm dữ liệu.

Hình được lưu ra `docs/confusion_matrix.png` để đưa vào báo cáo.

In [ ]:
cm = confusion_matrix(y_val, y_pred, labels=list(range(NUM_CLASSES)))

plt.figure(figsize=(7.5, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
)
plt.xlabel("Model doan")
plt.ylabel("Nhan that")
plt.title("Ma tran nham lan - validation chia theo burst")
plt.tight_layout()

CM_PNG_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(CM_PNG_PATH, dpi=150)
print("Da luu:", CM_PNG_PATH)
plt.show()

## Bước 7 — Độ chính xác từng lớp và cảnh báo cặp dễ nhầm

> **Đây là con số đưa vào báo cáo** — đo trên validation chia theo burst (Bước 3).
> Không dùng con số của bất kỳ lần chia ngẫu nhiên nào trước đây.

Ô cuối cùng trả lời đúng một câu hỏi: **cử chỉ nào chưa dùng được?**

Ngưỡng cảnh báo là **10%** — nghĩa là nếu cứ 10 lần làm cử chỉ A thì có hơn 1 lần
model đọc thành B, cặp (A, B) bị coi là chưa đạt.

Với hệ khoá cửa cần 3 cử chỉ liên tiếp, sai số dồn lại rất nhanh: mỗi cử chỉ đúng 90%
thì cả chuỗi chỉ còn `0.9³ ≈ 73%` — cứ 4 lần mở cửa là hỏng 1 lần.

**Cách xử lý khi có cảnh báo:**
1. Thu thêm 100–150 mẫu cho **cả hai** lớp trong cặp bị nhầm, cố ý làm ở nhiều góc
   tay và khoảng cách khác nhau, rồi train lại.
2. Nếu vẫn nhầm, **đổi mật mã** sang cặp cử chỉ khác — cách này nhanh và chắc ăn hơn.

In [ ]:
NGUONG_CANH_BAO = 0.10

print("DO CHINH XAC TUNG LOP (tren validation chia theo burst)")
print("-" * 46)
print(f"{'lop':<12}{'so mau':>8}{'dung':>7}{'do chinh xac':>15}")
print("-" * 46)
for i, ten in enumerate(CLASS_NAMES):
    tong = int(cm[i].sum())
    dung = int(cm[i, i])
    if tong == 0:
        print(f"{ten:<12}{0:>8}{'-':>7}{'khong co mau':>15}")
        continue
    acc = dung / tong
    danh_dau = "" if acc >= 0.95 else ("  <-- yeu" if acc >= 0.85 else "  <-- RAT YEU")
    print(f"{ten:<12}{tong:>8}{dung:>7}{acc:>14.1%}{danh_dau}")
print("-" * 46)

# --- Cac cap lop nham nhau qua nguong ---
canh_bao = []
for i in range(NUM_CLASSES):
    tong = int(cm[i].sum())
    if tong == 0:
        continue
    for j in range(NUM_CLASSES):
        if i == j:
            continue
        ty_le = cm[i, j] / tong
        if ty_le > NGUONG_CANH_BAO:
            canh_bao.append((ty_le, CLASS_NAMES[i], CLASS_NAMES[j], int(cm[i, j]), tong))

print()
if not canh_bao:
    print(f"OK - khong co cap lop nao nham nhau qua {NGUONG_CANH_BAO:.0%}.")
    print("Model du tot de chay gesture_lock.py.")
else:
    print(f"CANH BAO - co {len(canh_bao)} cap lop nham nhau tren {NGUONG_CANH_BAO:.0%}:")
    print()
    for ty_le, that, doan, so_lan, tong in sorted(canh_bao, reverse=True):
        print(f"  '{that}' bi doan thanh '{doan}': {so_lan}/{tong} = {ty_le:.1%}")
    print()
    print("Xu ly: thu them du lieu cho CA HAI lop trong moi cap roi train lai,")
    print("hoac doi mat ma sang cu chi khac (sua secret.default_code trong config.json")
    print("va bien MA_MAC_DINH trong middleware/flows.json).")

# --- Nhac rieng ve cac cu chi dang nam trong mat ma ---
ma = cfg["secret"]["default_code"]
hex_to_label = {g["hex"]: g["label"] for g in cfg["gestures"] if g["hex"]}
trong_ma = [hex_to_label.get(ma[k:k + 2]) for k in range(0, len(ma), 2)]
print()
print(f"Mat ma hien tai '{ma}' dung cac cu chi: {trong_ma}")
lien_quan = [c for c in canh_bao if c[1] in trong_ma or c[2] in trong_ma]
if lien_quan:
    print("NGHIEM TRONG: cac cu chi trong mat ma dang nam trong cap bi nham.")
else:
    print("Cac cu chi trong mat ma khong nam trong cap nham nao.")

## Bước 8 — Train lại trên toàn bộ dữ liệu (model triển khai)

> **Con số báo cáo là của lần chia nhóm ở Bước 3–7. Model triển khai là bản train lại
> trên toàn bộ dữ liệu ở bước này.** Hai thứ này khác nhau và phải ghi rõ như vậy trong báo cáo.

Validation chia theo burst giữ lại khoảng 20% số burst — đó là những tư thế tay mà model
chạy thật nên được học. Vì vậy sau khi đã đo xong, dựng một mạng mới bằng `dung_model()`
(cùng kiến trúc, cùng seed) và train trên **100% dữ liệu**, đúng `SO_EPOCH_TOT_NHAT` epoch
tìm được ở Bước 5.

Bản này **không có con số đánh giá riêng**: không còn mẫu nào nó chưa thấy. Không lấy độ
chính xác trên chính dữ liệu train của nó đưa vào báo cáo. Nó được kiểm bằng HaGRID
(`vision/eval_external.py`) và bằng phiên chạy thật (`scripts/dem_10_lan.py`).

In [ ]:
model_trien_khai = dung_model()
model_trien_khai.fit(
    X, y,
    epochs=SO_EPOCH_TOT_NHAT,
    batch_size=128,
    verbose=0,
)
print(f"Da train ban trien khai tren TOAN BO {len(y)} mau, {SO_EPOCH_TOT_NHAT} epoch.")
print("Con so bao cao: lay o Buoc 7 (validation chia theo burst), KHONG lay tu model nay.")

## Bước 9 — Lưu model triển khai

Lưu `model_trien_khai` (bản train toàn bộ ở Bước 8) ra **hai** định dạng:

- **`.keras`** — bản gốc đầy đủ, để sau này train tiếp hoặc phân tích.
- **`.tflite`** — bản rút gọn, đây mới là bản mà `gesture_lock.py` nạp lúc chạy thật.
  TFLite khởi động nhanh hơn và nhẹ hơn nhiều so với nạp cả TensorFlow.

Ô cuối **đối chiếu hai model trên cùng dữ liệu**. Bước này nhìn thừa nhưng rất cần:
nếu chuyển đổi TFLite hỏng, model vẫn chạy và vẫn trả về số, chỉ là trả về **sai** —
lúc đó rất khó lần ra vì `gesture_lock.py` sẽ chỉ đơn giản là đoán lung tung.

In [ ]:
KERAS_PATH.parent.mkdir(parents=True, exist_ok=True)
model_trien_khai.save(KERAS_PATH)
print("Da luu Keras :", KERAS_PATH, f"({KERAS_PATH.stat().st_size / 1024:.0f} KB)")

# Keras 3 khuyen dung duong export -> SavedModel -> TFLite.
# Van thu from_keras_model truoc vi no gon hon, hong thi quay ve cach kia.
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model_trien_khai)
    tflite_bytes = converter.convert()
except Exception as e:
    print("from_keras_model khong dung duoc:", type(e).__name__, "- chuyen sang SavedModel")
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        model_trien_khai.export(tmp)
        converter = tf.lite.TFLiteConverter.from_saved_model(tmp)
        tflite_bytes = converter.convert()

TFLITE_PATH.write_bytes(tflite_bytes)
print("Da luu TFLite:", TFLITE_PATH, f"({len(tflite_bytes) / 1024:.0f} KB)")

In [ ]:
# Doi chieu TFLite voi ban Keras TRIEN KHAI tren toan bo du lieu.
# (Khong so voi y_pred o Buoc 6 - do la du doan cua model danh gia, khong phai ban duoc luu.)
keras_pred = np.argmax(model_trien_khai.predict(X, verbose=0), axis=1)

interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

tflite_pred = []
for row in X:
    interpreter.set_tensor(inp["index"], row.reshape(1, -1).astype(np.float32))
    interpreter.invoke()
    tflite_pred.append(int(np.argmax(interpreter.get_tensor(out["index"])[0])))
tflite_pred = np.array(tflite_pred)

khop = float((tflite_pred == keras_pred).mean())
print(f"Ty le TFLite trung khop voi Keras trien khai: {khop:.4%} (tren {len(X)} mau)")
if khop < 0.999:
    print("CANH BAO: hai model lech nhau. KHONG dung file .tflite nay, kiem tra lai buoc chuyen doi.")
else:
    print("OK - file .tflite dung duoc cho gesture_lock.py")